当前大模型的 **post-training（后训练）**不是某一种算法，而是一整套把“基础模型”变成“可用助手或推理模型”的训练流程。

最常见的路线可以概括为：

预训练模型
   ↓
领域继续预训练（可选）
   ↓
SFT 指令微调
   ↓
偏好优化：RLHF / DPO / RLAIF
   ↓
推理强化：RLVR / PPO / GRPO
   ↓
安全、工具使用和专项能力训练
一、先区分几个容易混淆的概念
概念	解决什么问题	是否属于训练目标
SFT	教模型如何回答指令	是
RLHF	让回答更符合人类偏好	是
DPO	更简单地学习偏好数据	是
RLVR	用可验证答案提升推理	是
LoRA	用更少参数完成上述训练	否，它是训练方式
Quantization	降低模型推理或训练成本	否
RAG	在推理时检索外部知识	通常不属于 post-training

也就是说，LoRA可以用来做SFT或DPO；它和SFT、DPO不是同一层级的方法。

1. Continued Pre-training：继续预训练

基础模型完成通用预训练以后，继续使用特定领域的大量原始文本训练。

例如要做金融模型，可以继续训练：

财报
交易记录的文本描述
金融法规
分析报告
金融新闻

训练目标通常仍然是：

$$ \mathcal{L}_{CPT} = -\sum_t \log P_\theta(x_t\mid x_{<t}) $$

也就是根据前面的内容预测下一个token，和原始预训练基本一样。

它能学到什么？

它主要让模型获得：

专业术语
领域知识
特定语言风格
特定数据分布
代码语言或小语种能力
它不能直接教会什么？

它不能保证模型会：

按照用户指令回答
拒绝危险请求
给出固定格式
使用工具完成任务

因为训练数据通常只是原始文本，而不是“用户问题—助手回答”。

风险
灾难性遗忘：金融数据训练太多后，通用能力下降。
领域过拟合：什么问题都用金融风格回答。
数据污染：评测题或敏感信息混入训练集。

因此常把领域数据和一部分通用数据混合训练。

2. SFT：监督式指令微调

SFT，全称 Supervised Fine-Tuning，是最基础、最常见的后训练方法。

数据通常长这样：

User: 用简单语言解释什么是过拟合。
Assistant: 过拟合是指模型过度记住了训练数据……

训练时，模型看到问题和标准答案，然后最大化生成标准答案的概率：

$$ \mathcal{L}_{SFT} = -\sum_t \log P_\theta(y_t\mid x,y_{<t}) $$

其中：

\(x\)：用户问题
\(y\)：标准答案
\(y_t\)：答案中的第 \(t\) 个token
SFT本质是什么？

本质上是 behavior cloning（行为模仿）：

给模型看高质量回答，让模型模仿这些回答。

它可以教模型：

遵循指令
多轮对话
按JSON格式输出
写代码
调用工具
展示推理格式
拒绝危险请求

InstructGPT使用的经典流程就是先做SFT，再训练奖励模型，最后使用PPO优化。

SFT的优点
训练稳定
实现简单
数据形式直观
适合建立基础行为
可以用LoRA降低成本
SFT的缺点

它只知道“这个答案应该被模仿”，却不知道：

为什么答案A比答案B好
两个都能接受的答案谁更优
哪些错误特别严重
在多个目标之间如何权衡

而且模型只能模仿训练数据中已经出现过的能力。如果训练轨迹质量一般，模型也会学习其中的错误。

3. Rejection Sampling与自训练

这是一种很实用的“生成—筛选—再训练”方法。

流程是：

对每个问题生成多个候选答案。
使用人类、奖励模型、规则或答案验证器打分。
保留最好的答案。
用这些高质量答案继续做SFT。

例如一道数学题生成16个答案：

答案1：错误
答案2：正确
答案3：格式错误
……
答案12：正确且推理清楚

选择答案12作为新的SFT训练数据。

常见变体
Best-of-N sampling
Expert Iteration
Self-training
Self-improvement
Distillation
Rejection Sampling Fine-Tuning
优点
比完整RL简单稳定
可以大量利用模型自己生成的数据
适合数学、代码等可自动验证任务
可以把强模型能力蒸馏到小模型
缺点
很难学到模型目前完全生成不出来的策略
验证器有偏差，模型就会复制这种偏差
数据多样性容易降低
生成多个答案的成本较高
4. Reward Modeling：奖励模型

偏好数据通常不是一个标准答案，而是：

问题：解释什么是过拟合

回答A：准确、清楚、有例子
回答B：含糊，而且存在错误

人类选择：A > B

然后训练奖励模型：

$$ r_\phi(x,y) $$

让它给每个回答一个分数。

经典的pairwise loss是：

$$ \mathcal{L}_{RM} = -\log \sigma \left( r_\phi(x,y_w)-r_\phi(x,y_l) \right) $$

其中：

\(y_w\)：preferred/winner回答
\(y_l\)：rejected/loser回答
\(r_\phi\)：奖励模型给出的分数

目标是让：

$$ r_\phi(x,y_w)>r_\phi(x,y_l) $$

奖励模型学到的不是事实本身，而是“人类通常更喜欢什么样的回答”。

奖励模型可以评价
正确性
有用性
安全性
表达清晰度
是否遵循指令
是否包含必要信息
是否使用了正确格式

但奖励模型本身可能被模型利用，这就是 reward hacking。

5. RLHF：基于人类反馈的强化学习

RLHF通常由三部分组成：

SFT得到初始助手模型。
人类比较不同回答，训练奖励模型。
使用强化学习提高模型获得的奖励。

经典算法是 PPO。

模型生成回答：

$$ y\sim \pi_\theta(y\mid x) $$

奖励模型给出分数：

$$ r_\phi(x,y) $$

训练目标大致是：

$$ \max_\theta \mathbb{E}[r_\phi(x,y)] - \beta D_{KL}(\pi_\theta\|\pi_{\text{ref}}) $$

这里的KL项很重要：它限制训练后的模型不要偏离原始SFT模型太远。

为什么需要KL限制？

如果只最大化奖励，模型可能找到奖励模型的漏洞，例如：

回答异常冗长
反复使用奖励模型喜欢的词
表面礼貌但内容空洞
生成奇怪格式骗取高分
丧失语言流畅性
RLHF的优点
可以直接优化最终行为
能利用模型自己生成的新回答，即online learning
可以在复杂偏好之间做权衡
对开放式聊天任务比较合适
RLHF的缺点
需要单独训练奖励模型
PPO训练复杂、计算量大
对超参数敏感
容易出现reward hacking
人类标注成本高
奖励模型可能存在文化或群体偏见
6. DPO：直接偏好优化

DPO，即 Direct Preference Optimization。

它使用和RLHF相同的偏好数据：

prompt
chosen response
rejected response

但DPO：

不单独训练奖励模型
不运行PPO
直接提高chosen回答的相对概率
降低rejected回答的相对概率

可以简单理解为：

$$ P(y_w\mid x)\uparrow \qquad P(y_l\mid x)\downarrow $$

更准确地说，DPO比较当前模型和参考模型对两个回答的相对偏好。

DPO论文表明，它可以把标准RLHF目标转化成一个类似分类的损失函数，从而省去奖励模型和RL训练过程。

DPO的优点
实现比PPO简单
训练更稳定
计算成本更低
不需要在线生成回答
不需要显式奖励模型
非常适合开源模型微调
DPO的缺点
通常是offline learning，只能使用已有偏好数据
数据质量直接决定训练结果
对chosen和rejected之间的差异敏感
容易受到回答长度偏差影响
不能像在线RL一样持续探索新的策略
常见DPO变体
方法	核心变化
IPO	改进偏好目标，减少过度拟合
KTO	可以只使用“好/坏”标签，不一定要求成对答案
ORPO	把SFT和偏好优化结合在一个目标中
SimPO	不使用单独的参考模型
Online DPO	边生成新答案边更新偏好模型

例如SimPO使用平均token log probability作为隐式奖励，并取消参考模型，从而降低显存和计算成本。

7. RLAIF：基于AI反馈的强化学习

RLAIF把RLHF中的“人类评价”部分替换为AI评价。

流程可能是：

当前模型生成两个回答。
更强的模型或judge model比较回答。
生成偏好标签。
用这些偏好训练奖励模型或直接进行DPO。
再通过RL或偏好优化训练目标模型。
为什么使用AI反馈？

人类标注有几个问题：

成本高
速度慢
标注标准不一致
数学和代码问题需要专业人员
很难覆盖大量极端场景

AI可以更快地产生大规模偏好数据。

Constitutional AI

Constitutional AI是一种典型RLAIF方法：

先制定一组原则或“宪法”。
模型根据原则批评自己的回答。
模型重写回答。
使用修改后的回答进行SFT。
AI按照原则比较不同回答。
使用AI偏好进行RL训练。

该方法同时包含监督学习阶段和RLAIF阶段。

缺点
Judge model可能有系统性偏差
模型可能偏爱与自己风格相似的答案
AI评价不等于真实用户偏好
错误可能被大规模复制
多个模型可能共享相同盲点

因此实际训练中常把人类反馈、AI反馈和规则验证结合起来。

8. RLVR：基于可验证奖励的强化学习

这是近年来推理模型中非常重要的一类方法。

RLVR全称：

Reinforcement Learning with Verifiable Rewards

它不依赖“这个回答看起来好不好”的主观评价，而是使用可以自动验证的结果。

例如：

数学题
模型最终答案 = 42
标准答案 = 42
Reward = 1
编程题
代码通过全部单元测试
Reward = 1
SQL任务
SQL成功执行，而且结果正确
Reward = 1
工具调用
选择正确工具 + 参数正确 + 最终任务完成
Reward = 1

它的优势是奖励相对客观，不需要训练一个主观奖励模型。

9. PPO与GRPO
PPO

PPO在RLHF中很经典。它限制每次参数更新的幅度，避免模型因为一次高奖励样本变化过大。

但是训练LLM时，PPO通常需要：

Policy model
Reference model
Reward model
Value/Critic model

因此显存和工程成本较高。

GRPO

GRPO是 Group Relative Policy Optimization。

它对同一个问题生成一组答案：

$$ y_1,y_2,\ldots,y_G $$

分别获得奖励：

$$ r_1,r_2,\ldots,r_G $$

然后使用组内相对表现计算优势，例如：

$$ A_i = \frac{r_i-\text{mean}(r)} {\text{std}(r)+\epsilon} $$

高于组内平均水平的回答得到正优势，低于平均水平的得到负优势。

GRPO和PPO的重要区别

GRPO通常不需要单独训练value model，而是使用同一个问题下多个回答的相对奖励作为baseline，因此降低了显存需求。

GRPO最早在DeepSeekMath中被系统介绍，用于提升数学推理能力；论文将其描述为一种更节省PPO内存的变体。

GRPO适合什么任务？
数学推理
编程
逻辑题
有明确答案的问答
工具调用
可以自动检查结果的任务

DeepSeek-R1的研究进一步展示了RL训练能够促进自我检查、反思和策略调整等推理行为。

GRPO的风险
Reward只有0/1时非常稀疏
如果一组答案全部错误，学习信号很弱
可能只优化最终答案而不优化推理过程
模型可能利用验证器漏洞
输出可能越来越长
需要大量在线采样，推理成本很高
10. Outcome Supervision与Process Supervision

这两者区别在于奖励发生在哪里。

Outcome supervision：结果监督

只看最终答案：

步骤1 → 步骤2 → 错误步骤 → 碰巧得到正确答案
Reward = 1

问题是：模型可能推理过程错误，但最后猜对。

Process supervision：过程监督

对每个推理步骤分别评价：

步骤1：正确
步骤2：正确
步骤3：错误
步骤4：基于错误继续推导

可以训练 Process Reward Model，PRM：

$$ r_1,r_2,\ldots,r_T $$

让模型知道具体在哪一步出错。

过程监督在一些数学推理实验中明显优于只监督最终结果。

Process supervision的优点
更容易定位错误
可以减少“答案正确但过程错误”
适合数学、逻辑和多步规划
可以用于搜索时选择更好的推理分支
缺点
标注每一步非常昂贵
什么算“正确步骤”并不总是明确
可能限制模型探索新的推理路径
模型展示的文字推理不一定等于其内部计算过程
11. Distillation：蒸馏

蒸馏是让较小的student model模仿更强的teacher model。

例如：

Teacher：70B推理模型
Student：7B模型

Teacher生成：

高质量回答
数学解题轨迹
代码
工具调用轨迹
修改和批评
偏好标签

然后Student通过SFT或DPO学习这些数据。

优点
小模型获得部分大模型能力
推理成本更低
适合部署到本地或特定产品
不需要公开Teacher的权重
缺点
Student能力受到Teacher限制
Teacher错误会被复制
复杂推理轨迹不一定能被小模型完全学会
容易降低回答多样性
12. Tool-use与Agent post-training

工具使用本身不是一种独立优化算法，它是一种训练任务。

训练数据可能长这样：

User：旧金山今天的天气怎么样？
Assistant：<call_weather location="San Francisco">
Tool：18°C, cloudy
Assistant：旧金山今天18°C，多云。

常见训练方法包括：

用正确工具轨迹做SFT
用chosen/rejected工具轨迹做DPO
根据任务是否成功给予RL奖励
根据参数是否正确给予中间步骤奖励
用自动环境执行代码或API验证结果

训练目标可能包括：

是否选择了正确工具
工具参数是否正确
是否正确读取工具结果
是否避免虚构工具输出
多步骤任务最终是否完成
出错后能否恢复

对于Agent任务，奖励常常是稀疏的：只有整个任务完成才得到1分。因此也会加入步骤级奖励或课程学习。

13. Safety post-training

安全后训练通常不是一个单独阶段，而是混合使用多种方法：

安全示例SFT
拒绝回答数据
安全偏好DPO
红队攻击数据
Constitutional AI
安全奖励模型
对抗训练
越狱测试和迭代训练

一个关键问题是平衡：

$$ \text{Safety} \quad \text{vs.} \quad \text{Helpfulness} $$

训练不足会导致危险回答；训练过度则可能出现 over-refusal：正常问题也拒绝回答。

因此好的安全数据需要包含：

明确危险请求 → 拒绝
表面相似但安全的请求 → 正常回答
可以安全改写的问题 → 提供安全替代方案
不确定的问题 → 先澄清上下文
14. PEFT：LoRA和QLoRA

LoRA不是一种对齐目标，而是一种降低训练成本的方法。

传统全参数微调会更新整个权重矩阵 \(W\)。LoRA保持原权重不变，只学习一个低秩增量：

$$ W' = W + BA $$

其中：

\(A\) 和 \(B\) 是较小的矩阵
rank \(r\) 远小于原矩阵维度
需要训练的参数显著减少

QLoRA进一步把基础模型量化后再训练LoRA adapter。

LoRA/QLoRA可以用于：

SFT
DPO
Reward Model训练
部分RL训练
领域适配

它们改变的是“更新多少参数”，而不是“模型学习什么目标”。

15. 当前最常见的实际组合
普通聊天助手
Base Model
→ SFT
→ DPO或RLHF
→ Safety tuning
数学/代码推理模型
Base Model
→ Continued Pre-training
→ Reasoning SFT
→ RLVR（PPO或GRPO）
→ Rejection sampling
→ 再次SFT或DPO
企业领域模型
Base Model
→ 领域继续预训练（可选）
→ 领域指令SFT
→ 专家偏好DPO
→ 工具调用训练
→ 安全与合规训练
低预算开源项目
开源Base/Instruct Model
→ QLoRA SFT
→ QLoRA DPO

这是个人或小团队最容易落地的路线。

最核心的区别
方法	训练数据	核心目标	主要优点	主要问题
Continued pre-training	原始文本	学领域分布	增加领域知识	可能遗忘通用能力
SFT	问题＋标准答案	模仿优质回答	简单稳定	不会比较答案优劣
Rejection Sampling	多个生成答案＋筛选	模仿筛选后的答案	比RL简单	受限于现有生成能力
RLHF/PPO	人类偏好＋奖励模型	最大化人类偏好	可以在线探索	复杂、昂贵、不稳定
DPO	chosen/rejected答案	直接学习相对偏好	简单稳定	依赖离线数据
RLAIF	AI生成的偏好	学习AI评价	数据规模大	Judge可能有偏差
RLVR	可验证答案	提升推理成功率	奖励客观	仅适合可验证任务
Process supervision	步骤级标签	优化推理过程	可定位中间错误	标注昂贵
Distillation	强模型生成的数据	模仿Teacher	降低部署成本	会继承Teacher错误

一句话总结：

SFT教模型“应该怎么回答”，DPO/RLHF教模型“哪种回答更好”，RLVR教模型“怎样得到可验证的正确结果”，而LoRA只是让这些训练更便宜。
